# RV32I-MLA board control notebook
## Overlay-managed execution flow with native C driver delegation

This notebook documents the current software control path for the **rv32i-mla** platform on PYNQ.

The notebook intentionally assigns responsibilities to the two software layers as follows:

- **Python / Jupyter**
  - overlay loading
  - experiment orchestration
  - path management
  - structured execution of validation runs
  - presentation of results

- **Native C driver utility**
  - `/dev/mem` access
  - BRAM window reads and writes
  - instruction memory staging
  - instruction and data memory dumps
  - result collection from the shared memory map

This separation is important for two reasons. First, it removes duplicated low-level MMIO logic from the notebook. Second, it makes the control path visible and auditable for demonstration, grading, and later extension work.

The current bitstream exposes the shared BRAM window used by the CPU design, but the software-visible register block does **not** yet expose:

- processor run control
- completion signaling
- cycle counting
- accelerator mailbox registers

Accordingly, this notebook documents a **staged execution flow**:

1. load the overlay
2. validate the board-visible memory map through the C driver
3. stage a test program into instruction memory
4. execute the processor using the currently available external/manual method
5. collect and inspect the resulting data memory contents

This notebook is therefore the appropriate control document for the current integration stage of the project.

## Assumptions and repository layout

The notebook assumes the project tree is present on the PYNQ board under:

`/home/xilinx/jupyter_notebooks/rv32i-mla`

Within that tree, the relevant paths are expected to be:

- `sw/build/rv32i_mla_app` — native ARM/Linux control utility
- `sw/tests/artifacts/` — toolchain-generated RISC-V `.bin`, `.elf`, and `.dump` files
- `rv32i_mla_bd.bit` — bitstream
- `rv32i_mla_bd.hwh` — hardware handoff file

If the repository layout changes later, only the path configuration cell requires modification. The orchestration logic remains unchanged.

The naming convention intentionally distinguishes:

- repository and directory names using `rv32i-mla`
- C identifiers and utility names using `rv32i_mla`

That split is conventional and does not create ambiguity.

In [ ]:
from pathlib import Path
import subprocess
import shlex
import time
from typing import Iterable, Optional

from pynq import Overlay

PROJECT_ROOT = Path("/home/xilinx/jupyter_notebooks/rv32i-mla")
SW_DIR = PROJECT_ROOT / "sw"
APP_PATH = SW_DIR / "build" / "rv32i_mla_app"
TEST_ARTIFACTS_DIR = SW_DIR / "tests" / "artifacts"

BIT_PATH = PROJECT_ROOT / "rv32i_mla_bd.bit"
HWH_PATH = PROJECT_ROOT / "rv32i_mla_bd.hwh"

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("SW_DIR             =", SW_DIR)
print("APP_PATH           =", APP_PATH)
print("TEST_ARTIFACTS_DIR =", TEST_ARTIFACTS_DIR)
print("BIT_PATH           =", BIT_PATH)
print("HWH_PATH           =", HWH_PATH)

assert PROJECT_ROOT.exists(), f"Missing project root: {PROJECT_ROOT}"
assert SW_DIR.exists(), f"Missing software directory: {SW_DIR}"
assert APP_PATH.exists(), f"Missing native C utility: {APP_PATH}"
assert BIT_PATH.exists(), f"Missing bitstream: {BIT_PATH}"


## Subprocess execution helpers

The notebook interacts with the validated native control utility through `subprocess`.

This design choice is deliberate. At the current project stage, invoking the C utility as an external process is preferable to introducing Python bindings, `ctypes`, or direct notebook-side MMIO duplication. The resulting flow is easier to debug, easier to explain during evaluation, and closer to the actual deployment model used on the board.

Two helper functions are defined:

- `run_cmd(...)` executes an arbitrary shell command and prints both the command line and any captured output.
- `run_app(...)` invokes the native `rv32i_mla_app` utility with optional `sudo`.

The notebook therefore remains the orchestrator, while the driver remains the low-level authority on the memory interface.

In [ ]:
def run_cmd(
    cmd,
    *,
    check: bool = True,
    capture: bool = True,
    cwd: Optional[Path] = None,
):
    if isinstance(cmd, str):
        cmd_list = shlex.split(cmd)
    else:
        cmd_list = [str(x) for x in cmd]

    print("$", " ".join(shlex.quote(x) for x in cmd_list))

    result = subprocess.run(
        cmd_list,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=capture,
    )

    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

    return result


def run_app(*args, sudo: bool = True, check: bool = True):
    cmd = []
    if sudo:
        cmd.append("sudo")
    cmd.append(str(APP_PATH))
    cmd.extend(str(a) for a in args)
    return run_cmd(cmd, check=check, capture=True, cwd=SW_DIR)


## Software-visible hardware contract

The current control software expects a shared BRAM window with the following logical layout:

- `imem` begins at byte offset `0x0000`
- `dmem` begins at byte offset `0x0100`
- `mmio` begins at byte offset `0x0200`

The present C utility reports the currently compiled memory contract through the `info` command. That output is treated as the authoritative software-side description for the current bitstream.

At the current integration point, the expected feature status is:

- run control: absent
- cycle counter: absent
- accelerator mailbox: absent

This notebook therefore validates the contract before any test program is staged.

In [ ]:
IMEM_BASE = 0x0000
DMEM_BASE = 0x0100
MMIO_BASE = 0x0200

print(f"IMEM_BASE = 0x{IMEM_BASE:08X}")
print(f"DMEM_BASE = 0x{DMEM_BASE:08X}")
print(f"MMIO_BASE = 0x{MMIO_BASE:08X}")


## Overlay initialization

Overlay loading remains a notebook responsibility because it is part of the board management layer rather than the low-level BRAM driver.

The bitstream is loaded directly through PYNQ. A short delay is included after configuration to avoid race conditions with immediate follow-up access from software. This is conservative rather than strictly necessary, but it improves notebook behavior during demonstrations and repeated runs.

In [ ]:
print("Loading overlay...")
ol = Overlay(str(BIT_PATH))
time.sleep(0.5)
print("Overlay loaded.")
print("IP blocks exposed by overlay:")
for name in sorted(ol.ip_dict.keys()):
    print("  ", name)


## Driver-side contract validation

The following validation steps exercise the already-built native utility on the board.

- `info` reports the compiled physical base address, map size, and feature flags.
- `ps_sanity` verifies that the processor system can read and write the shared BRAM window through the native driver path.

These commands are not mere diagnostics; they validate that the notebook, the Linux execution environment, the deployed C utility, and the board-visible memory mapping are consistent with one another.

In [ ]:
run_app("info")
run_app("ps_sanity")


## Test artifact catalogue

The test programs used at this stage are external RISC-V binaries generated from the assembly validation suite.

The notebook treats the generated `.bin` files as immutable test inputs and resolves them from `sw/tests/artifacts/`. This avoids embedding instruction words directly into notebook cells and preserves traceability to the original assembly sources and build scripts.

The test catalogue below can be extended freely. The current entries correspond to the existing micro-diagnostic assembly programs.

In [ ]:
TESTS = {
    "smoke": {
        "bin": "smoke.bin",
        "description": "Minimal execute-and-store validation program.",
    },
    "no_branch": {
        "bin": "no_branch.bin",
        "description": "Straight-line execution path without branch or jump behavior.",
    },
    "branch_only": {
        "bin": "branch_only.bin",
        "description": "Branch-focused control-flow validation case.",
    },
    "jump_only": {
        "bin": "jump_only.bin",
        "description": "Jump-focused control-flow validation case.",
    },
}

for name, meta in TESTS.items():
    print(f"{name:12s} -> {meta['bin']:16s} | {meta['description']}")


## Path resolution and driver wrappers

The next helper set provides a thin notebook API over the C driver utility.

The intent is not to duplicate the driver's logic, but to present a stable orchestration vocabulary inside the notebook:

- `artifact_path(...)` resolves a named test binary
- `stage_test(...)` loads a selected binary into instruction memory
- `dump_imem(...)` prints instruction memory words through the driver
- `dump_dmem(...)` prints data memory words through the driver
- `show_results(...)` invokes the driver's result-reporting path

This keeps the notebook concise while preserving a very explicit execution trace.

In [ ]:
def artifact_path(name: str) -> Path:
    candidate = Path(name)
    if candidate.suffix == "":
        candidate = candidate.with_suffix(".bin")

    full_path = TEST_ARTIFACTS_DIR / candidate.name
    if not full_path.exists():
        raise FileNotFoundError(f"Artifact not found: {full_path}")
    return full_path


def stage_test(name: str):
    bin_path = artifact_path(name)
    print(f"Staging test binary: {bin_path}")
    run_app("stage_bin", str(bin_path))


def dump_imem(words: int = 32):
    run_app("dump_imem", str(words))


def dump_dmem(words: int = 32):
    run_app("dump_dmem", str(words))


def show_results():
    run_app("results")


## Example: stage a validation program

The following cell demonstrates the staging flow using one of the toolchain-generated test binaries.

At this stage of the project, staging is fully software-driven but execution itself is **not yet** exposed through a notebook-visible start/done control register set. Accordingly, the cell below performs only the loading and inspection portion of the flow.

The example may be changed to any of the named test programs in the catalogue above.

In [ ]:
stage_test("smoke")
dump_imem(16)


## Execution boundary

The currently deployed bitstream does not yet expose notebook-driven run control. This is a structural property of the present hardware/software contract, not a limitation of the notebook.

Accordingly, processor execution is still triggered through the already-established external or manual board-side method. Once the processor has completed the staged program, the next cells collect and inspect data memory state through the same validated C driver path.

This explicit boundary is useful in project documentation because it clearly distinguishes:

- what is already integrated and demonstrable
- what is intentionally deferred to the next RTL/system integration step

In [ ]:
print("Program image is staged in instruction memory.")
print("Execute the processor using the currently available external/manual method.")
print("After execution has completed, continue with the collection cells below.")


## Data memory and result collection

After processor execution, the notebook reuses the same driver path to inspect data memory and invoke the driver's result-reporting utility.

This is the preferred collection method for the current integration point because it guarantees that the same memory interface used for staging is also used for observation.

In [ ]:
dump_dmem(32)
show_results()


## Reusable orchestration helpers

The helpers below package the common board workflow into two high-level entry points:

- `run_test_until_execution_boundary(...)`
- `collect_test_results(...)`

This makes repeated demonstrations and comparisons substantially easier while preserving the explicit staged-execution model required by the current bitstream.

In [ ]:
def run_test_until_execution_boundary(test_name: str, imem_words: int = 16):
    if test_name not in TESTS:
        raise KeyError(f"Unknown test name: {test_name}")

    print(f"=== preparing test: {test_name} ===")
    print(TESTS[test_name]["description"])
    run_app("info")
    run_app("ps_sanity")
    stage_test(test_name)
    dump_imem(imem_words)
    print()
    print("Execution boundary reached.")
    print("Processor execution should now be triggered using the currently deployed board method.")


def collect_test_results(dmem_words: int = 32):
    print("=== collecting results ===")
    dump_dmem(dmem_words)
    show_results()


## Convenience wrappers for the existing assembly suite

These thin wrappers simplify repeated demonstration runs. They are intentionally straightforward and may be extended later with expected-value checking, run logs, or markdown summaries.

In [ ]:
def run_smoke():
    run_test_until_execution_boundary("smoke")


def run_no_branch():
    run_test_until_execution_boundary("no_branch")


def run_branch_only():
    run_test_until_execution_boundary("branch_only")


def run_jump_only():
    run_test_until_execution_boundary("jump_only")


## Optional repository presence check

This final cell is useful after directory refactors, file renames, or board-side re-deployment. It verifies that the notebook can still resolve the project assets required by the documented flow.

In [ ]:
for path in [PROJECT_ROOT, SW_DIR, APP_PATH, TEST_ARTIFACTS_DIR, BIT_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")


## Summary

This notebook documents the current integrated board workflow for the rv32i-mla project:

1. load the overlay from Python
2. validate the software-visible memory contract through the native C utility
3. stage instruction binaries through the same native utility
4. execute the CPU through the currently deployed external/manual mechanism
5. collect data memory and result output through the native utility

The notebook therefore serves both as an execution harness and as a formal integration document for the present project milestone.

A later revision can remove the explicit execution boundary once the RTL exposes software-visible run control, completion signaling, and any accelerator-specific control interface.